In [1]:
import pandas as pd
import os
from tqdm import tqdm

In [2]:
df_0030 = pd.read_csv("../datasets/schaefcomb_Wang2023SimpleGSR_ds000030.tsv", sep = "\t")

df_4302 = pd.read_csv("../datasets/schaefcomb_wang2023SimpleGSRCorrMatrix_ds004302.tsv", sep = "\t")

df_cobre = pd.read_csv("../datasets/schaefcomb_wang2023SimpleGSRCorrMatrix_cobre.tsv", sep = "\t")

/tmp/ipykernel_895109/508707093.py:5: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_cobre = pd.read_csv("../datasets/schaefcomb_wang2023SimpleGSRCorrMatrix_cobre.tsv", sep = "\t")


In [3]:
df_0030["group"] = df_0030["diagnosis"]

In [4]:
# Création de la colonne diagnosis
df_4302["diagnosis"] = df_4302["group"].map({
    "HC": "CONTROL",
    "AVH-": "SCHZ",
    "AVH+": "SCHZ",
})

df_4302["diagnosis"] = df_4302["diagnosis"].fillna("UNKNOWN")

# Renommer les valeurs de gender
df_4302["gender"] = df_4302["gender"].replace({
    "male": "M",
    "female": "F",
})

# Replacer la colonne diagnosis juste après "group"
cols = list(df_4302.columns)
group_idx = cols.index("group")
# retirer puis réinsérer à la bonne position
cols.insert(group_idx + 1, cols.pop(cols.index("diagnosis")))
df_4302 = df_4302[cols]


In [5]:
df_4302

,participant_id,group,diagnosis,age,gender,corr_1_2,corr_1_3,corr_1_4,corr_1_5,corr_1_6,...,corr_430_431,corr_430_432,corr_430_433,corr_430_434,corr_431_432,corr_431_433,corr_431_434,corr_432_433,corr_432_434,corr_433_434
0,sub-21,HC,CONTROL,42,M,0.260057,0.261104,-0.089561,-0.455227,-0.132276,...,NaN,-0.067477,0.255018,0.630073,NaN,NaN,NaN,0.129936,0.042653,0.272183
1,sub-10,HC,CONTROL,36,F,0.457691,0.132095,-0.037439,NaN,NaN,...,NaN,0.404580,0.328437,0.777075,NaN,NaN,NaN,0.117329,0.305994,0.348293
2,sub-29,AVH-,SCHZ,38,M,0.108464,0.382382,0.011199,0.113299,0.132078,...,NaN,0.154893,0.330102,0.639154,NaN,NaN,NaN,0.187120,0.059177,0.296410
3,sub-55,AVH+,SCHZ,44,M,-0.300822,0.181359,-0.085185,0.385769,0.079273,...,NaN,0.112297,0.307445,0.471354,NaN,NaN,NaN,0.259371,0.152311,0.445420
4,sub-13,HC,CONTROL,21,M,0.506121,0.067694,0.095354,NaN,NaN,...,NaN,0.440765,0.539905,0.742611,NaN,NaN,NaN,0.345999,0.425908,0.459046
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66,sub-02,HC,CONTROL,36,M,0.376694,-0.251911,0.110173,0.013275,-0.197859,...,NaN,-0.056809,-0.008535,0.600761,NaN,NaN,NaN,-0.075790,-0.094518,-0.174423
67,sub-23,HC,CONTROL,33,M,0.074220,NaN,-0.039888,NaN,NaN,...,NaN,0.166109,0.307369,NaN,NaN,NaN,NaN,-0.000363,NaN,NaN
68,sub-59,AVH+,SCHZ,25,M,-0.022304,NaN,0.268348,NaN,NaN,...,NaN,-0.060189,0.115147,0.795746,NaN,NaN,NaN,0.061766,-0.034630,0.189987
69,sub-66,AVH+,SCHZ,44,M,0.202535,0.372287,0.261661,0.062653,0.235857,...,NaN,-0.014756,0.190212,0.633174,NaN,NaN,NaN,-0.156721,-0.038832,0.202676


In [6]:
# Retirer group dans df_4302
df_4302 = df_4302.drop(columns=["group"])

# Ajouter la colonne site_id avant la concaténation
df_0030.loc[:, "site_id"] = "ds000030"
df_4302.loc[:, "site_id"] = "ds004302"

# Concaténer verticalement
df_open = pd.concat([df_0030, df_4302], ignore_index=True)

# Replacer site_id juste après diagnosis
cols = list(df_open.columns)
diag_idx = cols.index("diagnosis")
cols.insert(diag_idx + 1, cols.pop(cols.index("site_id")))
df_open = df_open[cols]


In [7]:
# Garder uniquement Control ou Patient
df_cobre = df_cobre[df_cobre["group"].isin(["Control", "Patient"])].copy()

# Remplacer les valeurs
df_cobre["group"] = df_cobre["group"].replace({
    "Control": "CONTROL",
    "Patient": "SCHZ"
})

# 3. Convertir age en float
df_cobre["age"] = df_cobre["age"].astype(float)

# 4. Ajouter colonne site
df_cobre["site_id"] = "ds_cobre"

# 5. Ajouter colonne diagnosis
df_cobre["diagnosis"] = df_cobre["group"].copy()

In [8]:
df_cobre

,participant_id,group,age,gender,corr_1_2,corr_1_3,corr_1_4,corr_1_5,corr_1_6,corr_1_7,...,corr_430_433,corr_430_434,corr_431_432,corr_431_433,corr_431_434,corr_432_433,corr_432_434,corr_433_434,site_id,diagnosis
1,sub-0040025,SCHZ,33.0,M,0.328032,0.589011,0.473182,0.224992,0.398275,0.383038,...,0.843858,0.831421,NaN,NaN,NaN,0.495688,0.425683,0.812936,ds_cobre,SCHZ
2,sub-0040079,SCHZ,26.0,M,0.510712,0.324923,-0.022333,-0.158258,0.118537,-0.248673,...,0.561737,0.519739,NaN,NaN,NaN,0.030595,-0.080179,0.587474,ds_cobre,SCHZ
3,sub-0040062,CONTROL,50.0,M,0.557951,0.541539,0.393446,0.423450,0.689149,-0.074260,...,0.531679,0.621323,NaN,NaN,NaN,0.354909,-0.106240,0.434653,ds_cobre,CONTROL
4,sub-0040110,SCHZ,43.0,M,0.167005,0.705872,-0.094055,0.133829,0.428239,0.094158,...,0.748208,0.747917,NaN,NaN,NaN,-0.103421,0.016269,0.786472,ds_cobre,SCHZ
5,sub-0040097,SCHZ,52.0,F,0.558480,0.508952,0.192424,0.118128,0.276193,0.042147,...,0.664043,0.814652,NaN,NaN,NaN,0.583803,0.190179,0.568703,ds_cobre,SCHZ
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
143,sub-0040109,SCHZ,33.0,M,0.509909,0.648256,0.368561,0.269486,0.339154,0.283008,...,0.569751,0.642109,NaN,NaN,NaN,-0.041065,-0.012960,0.331764,ds_cobre,SCHZ
144,sub-0040139,CONTROL,28.0,F,0.201015,0.659998,0.249428,0.447337,0.572041,0.099193,...,0.612552,0.790864,NaN,NaN,NaN,0.020178,-0.087654,0.675409,ds_cobre,CONTROL
145,sub-0040100,SCHZ,35.0,M,0.663684,-0.109535,-0.039656,-0.509296,-0.184351,-0.470632,...,0.879938,0.892022,NaN,NaN,NaN,-0.115466,-0.240822,0.893324,ds_cobre,SCHZ
146,sub-0040000,SCHZ,20.0,F,0.393050,0.256273,0.027250,0.107054,0.458977,0.338626,...,0.583226,0.610135,NaN,NaN,NaN,0.375861,0.268279,0.581998,ds_cobre,SCHZ


In [9]:
df_schizo = pd.concat([df_open, df_cobre], ignore_index=True)

In [10]:
# Normaliser diagnosis
df_schizo["diagnosis"] = df_schizo["diagnosis"].replace({
    "Control": "CONTROL",
    "control": "CONTROL"
})

# Normaliser gender
df_schizo["gender"] = df_schizo["gender"].replace({
    "Male": "M", "male": "M", "M": "M",
    "Female": "F", "female": "F", "F": "F"
})

In [11]:
df_schizo.to_csv("schaefcomb_Wang2023SimpleGSR_dfschizo.tsv", sep = "\t")

In [12]:
df_schizo

,participant_id,diagnosis,site_id,age,gender,corr_1_2,corr_1_3,corr_1_4,corr_1_5,corr_1_6,...,corr_430_432,corr_430_433,corr_430_434,corr_431_432,corr_431_433,corr_431_434,corr_432_433,corr_432_434,corr_433_434,group
0,sub-10882,CONTROL,ds000030,23.0,M,0.418489,0.606393,0.311871,0.302105,0.361551,...,-0.050728,0.564527,0.664548,NaN,NaN,NaN,0.117467,-0.099475,0.415247,CONTROL
1,sub-50060,SCHZ,ds000030,41.0,M,0.041613,0.273176,0.306790,0.377838,0.477112,...,0.198801,0.227180,0.739058,NaN,NaN,NaN,0.071222,0.095882,0.383683,SCHZ
2,sub-11050,CONTROL,ds000030,48.0,M,0.433502,0.559388,0.468267,0.366098,0.282578,...,0.124248,0.562367,0.782964,NaN,NaN,NaN,0.206201,-0.005463,0.459371,CONTROL
3,sub-10638,CONTROL,ds000030,25.0,F,0.622896,0.348602,0.457920,0.145553,0.555730,...,0.259503,0.458037,0.678141,NaN,NaN,NaN,0.288266,0.186826,0.460205,CONTROL
4,sub-60006,BIPOLAR,ds000030,22.0,M,0.424745,0.243551,0.235706,0.284769,0.287402,...,0.157549,0.334220,0.521278,NaN,NaN,NaN,0.340982,0.331975,0.623370,BIPOLAR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
473,sub-0040109,SCHZ,ds_cobre,33.0,M,0.509909,0.648256,0.368561,0.269486,0.339154,...,0.001326,0.569751,0.642109,NaN,NaN,NaN,-0.041065,-0.012960,0.331764,SCHZ
474,sub-0040139,CONTROL,ds_cobre,28.0,F,0.201015,0.659998,0.249428,0.447337,0.572041,...,0.083698,0.612552,0.790864,NaN,NaN,NaN,0.020178,-0.087654,0.675409,CONTROL
475,sub-0040100,SCHZ,ds_cobre,35.0,M,0.663684,-0.109535,-0.039656,-0.509296,-0.184351,...,-0.054443,0.879938,0.892022,NaN,NaN,NaN,-0.115466,-0.240822,0.893324,SCHZ
476,sub-0040000,SCHZ,ds_cobre,20.0,F,0.393050,0.256273,0.027250,0.107054,0.458977,...,0.361932,0.583226,0.610135,NaN,NaN,NaN,0.375861,0.268279,0.581998,SCHZ
